# Bias Analysis Notebook

This notebook analyzes fairness metrics.



In [1]:
import pandas as pd

df = pd.read_csv("fairvisinput_clustered.csv")
df.head()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,HasMortgage,...,LoanPurpose_home,LoanPurpose_other,InterestLevel_medium,InterestLevel_high,DTIBucket_medium,DTIBucket_high,DTIBucket_extreme,y_true,y_pred,cluster
0,30,143756,106703,663,3,3,17.37,60,0.81,1,...,1,0,0,1,0,0,0,0,0,6
1,65,30348,151148,618,98,4,17.77,60,0.44,0,...,0,0,0,1,0,0,0,0,0,0
2,37,30104,228096,810,64,1,3.63,48,0.52,0,...,0,1,0,0,0,0,0,0,0,25
3,54,131336,130974,594,62,4,14.79,60,0.26,0,...,0,0,1,0,0,0,0,0,0,19
4,63,38176,52978,370,2,3,15.49,60,0.84,1,...,0,0,1,0,0,0,0,0,0,21


In [5]:

acc = (df['y_true'] == df['y_pred']).mean()
FN = ((df['y_true']==1) & (df['y_pred']==0)).sum()
TP = ((df['y_true']==1) & (df['y_pred']==1)).sum()
FNR = FN / (FN + TP + 1e-9)

FP = ((df['y_true']==0) & (df['y_pred']==1)).sum()
TN = ((df['y_true']==0) & (df['y_pred']==0)).sum()
FPR = FP / (FP + TN + 1e-9)

print("accuracy rate : ",acc)
print("False Negative rate : ",FNR)
print("False Positive rate : ",FPR)


accuracy rate :  0.9999366828125494
False Negative rate :  0.0005417851821751941
False Positive rate :  0.0


# Bias for ALL Features Automatically

In [6]:
categorical_features = [
    c for c in df.columns
    if df[c].nunique() <= 10 and c not in ['y_true', 'y_pred']
]

bias_table = []

for feat in categorical_features:
    groups = df.groupby(feat)
    for value, g in groups:
        bias_table.append({
            'feature': feat,
            'value': value,
            'size': len(g),
            'accuracy': (g['y_true']==g['y_pred']).mean(),
            'FNR': ((g['y_true']==1) & (g['y_pred']==0)).sum() / ((g['y_true']==1).sum() + 1e-9),
            'FPR': ((g['y_true']==0) & (g['y_pred']==1)).sum() / ((g['y_true']==0).sum() + 1e-9),
        })

bias_df = pd.DataFrame(bias_table)
bias_df.sort_values("FNR", ascending=False).head(20)


,feature,value,size,accuracy,FNR,FPR
8,LoanTerm,60,12586,0.999682,0.002701,0.0
32,MaritalStatus_married,1,21029,0.999857,0.001372,0.0
36,LoanPurpose_business,1,12727,0.999843,0.001281,0.0
14,HasCoSigner,1,31788,0.999874,0.001201,0.0
22,Education_master's,1,15763,0.999873,0.001157,0.0
10,HasMortgage,1,31627,0.999874,0.001135,0.0
3,NumCreditLines,4,15777,0.999873,0.000948,0.0
30,EmploymentType_unemployed,1,15916,0.999874,0.000916,0.0
12,HasDependents,1,31555,0.999905,0.000901,0.0
33,MaritalStatus_single,0,42093,0.999905,0.000825,0.0


**⭐ Overall Model Fairness

The model demonstrates exceptionally high accuracy across the entire dataset:

Global accuracy: ~99.99%

False Negative Rate (FNR): Extremely low across all groups

False Positive Rate (FPR): 0 for all groups

No group exhibits strong signs of systematic discrimination or disparate treatment.

👉 Conclusion:
The model is fair and highly consistent across demographic, credit, employment, and loan-purpose segments.
No critical bias detected. **